# 🔐 Smart Contract Vulnerability Detector — Hierarchical MIL
### GraphCodeBERT/CodeBERT + LoRA + **Gated-Attention Multiple-Instance-Learning**

Phiên bản nâng cấp cho `detect_dataset.jsonl` (8.218 mẫu, ~51.5% Vulnerable / 48.5% Safe).

**Vì sao bản cũ chưa hiệu quả & bản này sửa gì:**

| Vấn đề bản cũ | Bản mới |
|---|---|
| Sliding-window gán nhãn `Vulnerable` cho **mọi** chunk → nhiễu nhãn, Precision(Vuln) chỉ 0.53 | **MIL attention-pooling**: 1 nhãn / 1 contract; model chỉ cần "chú ý" đúng chunk chứa lỗi → hết nhiễu |
| Truncate 409 token (mất 74% thông tin vì đa số code >512 token) | Nhìn **toàn contract** qua nhiều chunk; contract siêu dài → chọn chunk **đều** (đầu/giữa/cuối) |
| 5 "prompt variant" + majority voting (rác với encoder) | Bỏ hẳn — đưa code trực tiếp; thay bằng **threshold calibration** trên val |
| Class-weight cực đoan (cho lệch 5:1) | Data đã cân bằng → class-weight nhẹ + **label smoothing** |
| Không chống rò rỉ | Split theo **nhóm cấu trúc** (contract đổi tên biến vẫn cùng split) |
| `microsoft/codebert-base` | **`microsoft/graphcodebert-base`** (mạnh hơn cho code; đổi 1 dòng nếu muốn quay lại) |

> **Kaggle:** Add Data → upload `detect_dataset.jsonl`. Notebook tự tìm file. Bật **GPU**. Dùng **1 GPU** (xem ghi chú ở cell cấu hình).

## ⚙️ 0. Cài đặt thư viện

> 🔴 **QUAN TRỌNG — chạy cell này TRƯỚC, rồi RESTART KERNEL, rồi Run All.**
> Bản `transformers 5.x` mặc định trên Kaggle đang lỗi deps (tokenizer converter + `@strict` RobertaConfig). Ta ghim về bộ **4.x ổn định, đã kiểm chứng**.

In [1]:
# Ghim bộ thư viện tương thích (tránh transformers 5.x lỗi trên Kaggle).
!pip install -q "transformers==4.46.3" "tokenizers==0.20.3" "huggingface_hub==0.25.2" "peft==0.13.2" "accelerate==1.0.1" "datasets==3.0.1" scikit-learn sentencepiece
import importlib
print("=== Phiên bản thư viện ===")
for pkg in ["transformers", "tokenizers", "huggingface_hub", "peft", "accelerate", "datasets", "torch"]:
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg}: {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"  {pkg}: !! {e}")
print("\n🔴 BẮT BUỘC: Menu Run -> Restart & Clear Cell Outputs (hoặc Restart Kernel), sau đó Run All.")
print("   (Vì transformers 5.x đã nạp sẵn trong session — phải restart mới nhận bản 4.46.)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026

2026-07-01 00:12:24.590105: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782864744.999621      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782864745.120284      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782864746.101917      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782864746.101968      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782864746.101971      23 computation_placer.cc:177] computation placer alr

  peft: 0.13.2
  accelerate: 1.0.1
  datasets: 3.0.1
  torch: 2.10.0+cu128

🔴 BẮT BUỘC: Menu Run -> Restart & Clear Cell Outputs (hoặc Restart Kernel), sau đó Run All.
   (Vì transformers 5.x đã nạp sẵn trong session — phải restart mới nhận bản 4.46.)


## 📦 1. Cấu hình

In [2]:
import os
# ⚠️ Model dùng collator tuỳ chỉnh (gộp chunk theo contract) -> KHÔNG tương thích nn.DataParallel.
# Vì vậy ép dùng 1 GPU cho ổn định (kể cả khi Kaggle cấp T4 x2). 1 T4 16GB là đủ cho dataset này.
os.environ["CUDA_VISIBLE_DEVICES"]  = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json, re, glob, hashlib, random, collections
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F

import transformers
_tv = int(transformers.__version__.split(".")[0])
assert _tv < 5, (f"transformers=={transformers.__version__}: bạn CHƯA restart kernel sau cell cài đặt! "
                 "Hãy Restart Kernel rồi Run All (cần bản 4.46).")
print("transformers:", transformers.__version__)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ----------------------- HYPER-PARAMS -----------------------
MODEL_NAME   = "microsoft/codebert-base"        # default an toàn (đã chạy được trong env của bạn)
# 💡 Nâng chất lượng: đổi MODEL_NAME = "microsoft/graphcodebert-base" (mạnh hơn cho code).
#    GIỮ NGUYÊN TOKENIZER_NAME bên dưới — graphcodebert dùng đúng vocab RoBERTa của codebert,
#    và file tokenizer của graphcodebert hay lỗi trên transformers 5.x.
TOKENIZER_NAME = "roberta-base"   # ⚠️ CÓ SẴN tokenizer.json -> nạp thẳng trên transformers 5.x.
#   CodeBERT & GraphCodeBERT đều khởi tạo từ roberta-base, CÙNG vocab BPE (50265, <s>=0/<pad>=1/</s>=2),
#   nên tokenizer roberta-base tương thích id 1:1 với cả hai model. Tránh lỗi converter của transformers 5.x.
MAX_LEN      = 512        # token mỗi chunk (BERT-based)
MAX_CHUNKS   = 8          # số chunk tối đa / contract (8×510 ≈ 4080 token code)
LABEL2ID     = {"Safe": 0, "Vulnerable": 1}
ID2LABEL     = {0: "Safe", 1: "Vulnerable"}

USE_LORA        = True
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
GRAD_CHECKPOINT = True     # tiết kiệm VRAM

# Loss
LABEL_SMOOTHING  = 0.05
USE_FOCAL        = False   # bật nếu muốn nhấn mạnh mẫu khó
FOCAL_GAMMA      = 2.0
USE_CLASS_WEIGHT = True    # nhẹ vì data ~cân bằng

# Train  (nếu OOM: giảm TRAIN_BS xuống 2, hoặc MAX_CHUNKS xuống 6/4)
EPOCHS     = 15
TRAIN_BS   = 4             # số CONTRACT / batch
EVAL_BS    = 8
GRAD_ACCUM = 4             # batch hiệu dụng = 16 contract
LR         = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

OUTPUT_DIR = "./vuln-hier"
SAVE_DIR   = "./vuln-hier-final"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Model :", MODEL_NAME)

transformers: 4.46.3
Device: Tesla T4
Model : microsoft/codebert-base


## 📂 2. Nạp dữ liệu
`detect_dataset.jsonl` đã ở dạng `{"code", "label"}` sạch (đã bỏ comment, chuẩn hoá nhãn) — không cần làm sạch thêm.

In [3]:
def find_data_path():
    hits = sorted(glob.glob("/kaggle/input/**/detect_dataset.jsonl", recursive=True))
    if hits:
        return hits[0]
    for p in ["detect_dataset.jsonl",
              "data/clean_dataset/detect_dataset.jsonl",
              "/kaggle/input/detect_dataset.jsonl"]:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("Không thấy detect_dataset.jsonl. Hãy 'Add Data' lên Kaggle rồi chạy lại.")

DATA_PATH = find_data_path()
print("DATA_PATH =", DATA_PATH)

rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8") if l.strip()]
rows = [r for r in rows if r.get("code") and r.get("label") in LABEL2ID]
n  = len(rows)
nv = sum(r["label"] == "Vulnerable" for r in rows)
print(f"Tổng mẫu: {n:,} | Vulnerable: {nv:,} ({100*nv/n:.1f}%) | Safe: {n-nv:,} ({100*(n-nv)/n:.1f}%)")
print("Ví dụ:", rows[0]["label"], "|", rows[0]["code"][:110].replace(chr(10), " "))

DATA_PATH = /kaggle/input/datasets/thanhphuocjr/smartbug-solodit-dappscan/detect_dataset.jsonl
Tổng mẫu: 8,218 | Vulnerable: 4,235 (51.5%) | Safe: 3,983 (48.5%)
Ví dụ: Vulnerable | pragma solidity ^0.4.11;  contract Ownable {   address public owner;    function Ownable() {     owner = msg.s


## ✂️ 3. Chia tập chống rò rỉ (group-aware)
Nhiều contract chỉ khác nhau tên biến (boilerplate OpenZeppelin…). Nếu bản gần-trùng nằm cả ở train lẫn test → **điểm số ảo cao**. Ta gom theo *chữ ký cấu trúc* (xoá hết định danh) và cho **cả nhóm** vào cùng một tập.

In [4]:
def structural_sig(code: str) -> str:
    s = re.sub(r"\b[A-Za-z_]\w*\b", "X", code)   # mask mọi tên biến/hàm/contract -> chỉ giữ cấu trúc
    s = re.sub(r"\s+", "", s)
    return hashlib.md5(s.encode("utf-8", "replace")).hexdigest()

groups = collections.defaultdict(list)
for i, r in enumerate(rows):
    groups[structural_sig(r["code"])].append(i)

gkeys = list(groups.keys())
random.Random(SEED).shuffle(gkeys)

n_test_target = int(0.10 * n)
n_val_target  = int(0.10 * n)
test_idx, val_idx, train_idx = [], [], []
for k in gkeys:
    g = groups[k]
    if len(test_idx) + len(g) <= n_test_target:
        test_idx += g
    elif len(val_idx) + len(g) <= n_val_target:
        val_idx += g
    else:
        train_idx += g

take = lambda idxs: [rows[i] for i in idxs]
train_raw, val_raw, test_raw = take(train_idx), take(val_idx), take(test_idx)

def bal(name, d):
    v = sum(x["label"] == "Vulnerable" for x in d)
    print(f"  {name:5s}: {len(d):5,d} | Vuln {v:5,d} ({100*v/max(1,len(d)):.1f}%)  Safe {len(d)-v:5,d}")
print(f"Số nhóm cấu trúc: {len(gkeys):,}  (so với {n:,} mẫu)")
bal("train", train_raw); bal("val", val_raw); bal("test", test_raw)

Số nhóm cấu trúc: 7,766  (so với 8,218 mẫu)
  train: 6,576 | Vuln 3,379 (51.4%)  Safe 3,197
  val  :   821 | Vuln   431 (52.5%)  Safe   390
  test :   821 | Vuln   425 (51.8%)  Safe   396


## 🪟 4. Tokenizer & Chunk hoá (MIL bags)
Mỗi contract → danh sách chunk ≤512 token. Contract dài hơn `MAX_CHUNKS` → **chọn chunk đều nhau** (phủ đầu/giữa/cuối) thay vì cắt cụt phần đuôi. Mỗi contract = một "bag" các chunk, **giữ 1 nhãn duy nhất**.

In [5]:
import importlib, subprocess, sys
# Đảm bảo có sẵn converter (transformers 5.x cần khi phải convert tokenizer) — cài inline, không cần restart.
for _p in ["sentencepiece", "tiktoken"]:
    try:
        importlib.import_module(_p)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _p], check=False)

from transformers import AutoTokenizer

def load_tokenizer(name):
    try:
        return AutoTokenizer.from_pretrained(name, use_fast=True)
    except Exception as e:
        print("Fast lỗi -> thử slow:", str(e)[:120])
        return AutoTokenizer.from_pretrained(name, use_fast=False)

tokenizer = load_tokenizer(TOKENIZER_NAME)
assert tokenizer.vocab_size >= 50000, f"Vocab {tokenizer.vocab_size} không khớp RoBERTa/CodeBERT!"

# transformers 5.x nạp từ tokenizer.json có thể KHÔNG gán sẵn vai trò special-token
# (cls/sep/pad = None). Gán lại theo chuẩn RoBERTa và lấy id trực tiếp từ vocab.
for attr, tok in [("bos_token","<s>"),("eos_token","</s>"),("unk_token","<unk>"),
                  ("pad_token","<pad>"),("cls_token","<s>"),("sep_token","</s>"),("mask_token","<mask>")]:
    if getattr(tokenizer, attr, None) is None:
        try: setattr(tokenizer, attr, tok)
        except Exception: pass

CLS = tokenizer.convert_tokens_to_ids("<s>")
SEP = tokenizer.convert_tokens_to_ids("</s>")
PAD = tokenizer.convert_tokens_to_ids("<pad>")
assert all(isinstance(x, int) and x >= 0 for x in (CLS, SEP, PAD)), f"special ids sai: {CLS}/{SEP}/{PAD}"
CONTENT = MAX_LEN - 2
print(f"Tokenizer OK | vocab={tokenizer.vocab_size:,} | CLS/SEP/PAD = {CLS}/{SEP}/{PAD}")

def chunk_encode(code: str):
    """code -> (input_ids[c,L], attention_mask[c,L]); c <= MAX_CHUNKS."""
    ids = tokenizer.encode(code, add_special_tokens=False)
    if len(ids) == 0:
        ids = [PAD]
    windows = [ids[i:i + CONTENT] for i in range(0, len(ids), CONTENT)]
    if len(windows) > MAX_CHUNKS:                       # chọn đều thay vì cắt đuôi
        sel = sorted(set(np.linspace(0, len(windows) - 1, MAX_CHUNKS).round().astype(int).tolist()))
        windows = [windows[i] for i in sel]
    input_ids, attn = [], []
    for w in windows:
        seq = [CLS] + w + [SEP]
        m   = [1] * len(seq)
        if len(seq) < MAX_LEN:
            pad = MAX_LEN - len(seq)
            seq += [PAD] * pad; m += [0] * pad
        input_ids.append(seq); attn.append(m)
    return torch.tensor(input_ids, dtype=torch.long), torch.tensor(attn, dtype=torch.long)

class ChunkedDS(torch.utils.data.Dataset):
    def __init__(self, items):
        self.data = []
        for it in items:
            iid, am = chunk_encode(it["code"])
            self.data.append({"input_ids": iid, "attention_mask": am, "label": LABEL2ID[it["label"]]})
    def __len__(self):       return len(self.data)
    def __getitem__(self, i): return self.data[i]

print("Đang chunk hoá... (vài phút)")
train_ds, val_ds, test_ds = ChunkedDS(train_raw), ChunkedDS(val_raw), ChunkedDS(test_raw)
tc = sum(d["input_ids"].shape[0] for d in train_ds.data)
print(f"Train: {len(train_ds):,} contract -> {tc:,} chunk (TB {tc/len(train_ds):.2f} chunk/contract)")

def collate(features):
    input_ids      = torch.cat([f["input_ids"] for f in features], 0)
    attention_mask = torch.cat([f["attention_mask"] for f in features], 0)
    n_chunks       = torch.tensor([f["input_ids"].shape[0] for f in features], dtype=torch.long)
    labels         = torch.tensor([f["label"] for f in features], dtype=torch.long)
    return {"input_ids": input_ids, "attention_mask": attention_mask,
            "n_chunks": n_chunks, "labels": labels}

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (529 > 512). Running this sequence through the model will result in indexing errors


Tokenizer OK | vocab=50,265 | CLS/SEP/PAD = 0/2/1
Đang chunk hoá... (vài phút)
Train: 6,576 contract -> 27,180 chunk (TB 4.13 chunk/contract)


## 🤖 5. Mô hình Hierarchical Gated-Attention (MIL)
Encoder (chia sẻ trọng số) encode từng chunk → lấy vector `[CLS]`. **Gated attention** (Ilse et al., 2018) học trọng số cho từng chunk rồi pooling thành 1 vector contract → head phân loại. Lỗ hổng nằm ở 1 chunk → attention dồn vào đó, không ép các chunk khác mang nhãn Vulnerable.

In [6]:
from transformers import AutoModel
from transformers.modeling_outputs import SequenceClassifierOutput
from peft import LoraConfig, get_peft_model, TaskType

class HierAttnClassifier(nn.Module):
    def __init__(self, encoder, hidden, num_labels=2, dropout=0.1,
                 class_weights=None, label_smoothing=0.0, use_focal=False, focal_gamma=2.0):
        super().__init__()
        self.encoder    = encoder
        self.attn_V     = nn.Linear(hidden, 128)
        self.attn_U     = nn.Linear(hidden, 128)
        self.attn_w     = nn.Linear(128, 1)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)
        self.num_labels = num_labels
        self.label_smoothing = label_smoothing
        self.use_focal  = use_focal
        self.focal_gamma = focal_gamma
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights)
        else:
            self.class_weights = None

    def forward(self, input_ids, attention_mask, n_chunks, labels=None):
        enc = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h   = enc.last_hidden_state[:, 0]                  # [N_chunks, H]  (CLS mỗi chunk)
        A   = self.attn_w(torch.tanh(self.attn_V(h)) * torch.sigmoid(self.attn_U(h)))  # [N_chunks,1]
        pooled, idx = [], 0
        for c in n_chunks:                                 # gộp theo từng contract
            c  = int(c)
            hi = h[idx:idx + c]                            # [c, H]
            ai = torch.softmax(A[idx:idx + c], dim=0)      # attention chuẩn hoá trong contract
            pooled.append((ai * hi).sum(0))               # [H]
            idx += c
        pooled = torch.stack(pooled, 0)                    # [B, H]
        logits = self.classifier(self.dropout(pooled))     # [B, num_labels]
        loss = None
        if labels is not None:
            w = self.class_weights if self.class_weights is not None else None
            if self.use_focal:
                ce = F.cross_entropy(logits, labels, weight=w, reduction="none")
                pt = torch.exp(-ce)
                loss = ((1 - pt) ** self.focal_gamma * ce).mean()
            else:
                loss = F.cross_entropy(logits, labels, weight=w, label_smoothing=self.label_smoothing)
        return SequenceClassifierOutput(loss=loss, logits=logits)

# ---- build ----
encoder = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
HIDDEN  = encoder.config.hidden_size
if USE_LORA:
    encoder = get_peft_model(encoder, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
        task_type=TaskType.FEATURE_EXTRACTION, target_modules=["query", "key", "value"]))
    encoder.print_trainable_parameters()
if GRAD_CHECKPOINT:
    try:
        encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        encoder.enable_input_require_grads()
        print("Gradient checkpointing: ON")
    except Exception as e:
        print("Gradient checkpointing OFF:", e)

cnt = collections.Counter(d["label"] for d in train_ds.data)
N   = sum(cnt.values())
cw  = torch.tensor([N / (2 * cnt[0]), N / (2 * cnt[1])], dtype=torch.float) if USE_CLASS_WEIGHT else None
print("Class weights:", cw)

model = HierAttnClassifier(encoder, HIDDEN, num_labels=2, dropout=0.1,
                           class_weights=cw, label_smoothing=LABEL_SMOOTHING,
                           use_focal=USE_FOCAL, focal_gamma=FOCAL_GAMMA).to(device)
n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model sẵn sàng | trainable params: {n_tr:,}")

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

trainable params: 884,736 || all params: 125,530,368 || trainable%: 0.7048
Gradient checkpointing: ON
Class weights: tensor([1.0285, 0.9731])
✅ Model sẵn sàng | trainable params: 1,083,267


## 🏋️ 6. Huấn luyện (LoRA + cosine + early stopping)

In [7]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import f1_score

def _logits(pred):
    return pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions

def compute_metrics(pred):
    logits = _logits(pred); labels = pred.label_ids
    preds  = np.argmax(logits, axis=-1)
    return {"f1_macro": f1_score(labels, preds, average="macro"),
            "f1_vuln":  f1_score(labels, preds, pos_label=1, average="binary", zero_division=0)}

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    label_names=["labels"],
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collate, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print("🚀 Bắt đầu huấn luyện...")
trainer.train()
print("✅ Huấn luyện hoàn tất!")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

🚀 Bắt đầu huấn luyện...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Vuln
0,0.642200,0.623718,0.669688,0.719577
2,0.571100,0.579927,0.733184,0.751708
4,0.551100,0.556581,0.745966,0.774123
6,0.521700,0.584720,0.741068,0.727506


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

✅ Huấn luyện hoàn tất!


## 📊 7. Hiệu chỉnh ngưỡng & Đánh giá
Thay cho majority-voting: tìm **ngưỡng quyết định tối ưu trên VAL** (tối đa hoá F1-macro) rồi áp lên TEST. Báo cáo cả ngưỡng 0.5 và ngưỡng đã hiệu chỉnh, kèm ROC-AUC / PR-AUC / confusion matrix.

In [8]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score, f1_score)

def get_probs(ds):
    out    = trainer.predict(ds)
    logits = out.predictions[0] if isinstance(out.predictions, tuple) else out.predictions
    probs  = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return probs, out.label_ids

val_probs, val_labels = get_probs(val_ds)
best_t, best_f1 = 0.5, -1.0
for t in np.linspace(0.05, 0.95, 91):
    f1m = f1_score(val_labels, (val_probs >= t).astype(int), average="macro")
    if f1m > best_f1:
        best_f1, best_t = f1m, float(t)
print(f"Ngưỡng tối ưu (VAL) = {best_t:.3f} | F1-macro(val) = {best_f1:.4f}")

test_probs, test_labels = get_probs(test_ds)
for name, t in [("0.50 (mặc định)", 0.5), (f"{best_t:.3f} (hiệu chỉnh)", best_t)]:
    preds = (test_probs >= t).astype(int)
    print("\n" + "=" * 62)
    print(f"📊 TEST @ threshold {name}")
    print("=" * 62)
    print(classification_report(test_labels, preds, target_names=["Safe", "Vulnerable"],
                                digits=4, zero_division=0))
    print("Confusion [[TN FP][FN TP]]:\n", confusion_matrix(test_labels, preds))
print(f"\nROC-AUC = {roc_auc_score(test_labels, test_probs):.4f} | "
      f"PR-AUC = {average_precision_score(test_labels, test_probs):.4f}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Ngưỡng tối ưu (VAL) = 0.560 | F1-macro(val) = 0.7563


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



📊 TEST @ threshold 0.50 (mặc định)
              precision    recall  f1-score   support

        Safe     0.7622    0.6313    0.6906       396
  Vulnerable     0.7039    0.8165    0.7560       425

    accuracy                         0.7272       821
   macro avg     0.7330    0.7239    0.7233       821
weighted avg     0.7320    0.7272    0.7245       821

Confusion [[TN FP][FN TP]]:
 [[250 146]
 [ 78 347]]

📊 TEST @ threshold 0.560 (hiệu chỉnh)
              precision    recall  f1-score   support

        Safe     0.7500    0.6667    0.7059       396
  Vulnerable     0.7186    0.7929    0.7539       425

    accuracy                         0.7320       821
   macro avg     0.7343    0.7298    0.7299       821
weighted avg     0.7337    0.7320    0.7307       821

Confusion [[TN FP][FN TP]]:
 [[264 132]
 [ 88 337]]

ROC-AUC = 0.7998 | PR-AUC = 0.7716


## 💾 8. Lưu mô hình

In [9]:
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "model_state.pt"))
tokenizer.save_pretrained(SAVE_DIR)
json.dump({"model_name": MODEL_NAME, "max_len": MAX_LEN, "max_chunks": MAX_CHUNKS,
           "hidden": HIDDEN, "use_lora": USE_LORA, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
           "threshold": best_t, "label2id": LABEL2ID},
          open(os.path.join(SAVE_DIR, "infer_config.json"), "w"), indent=2)
print("Đã lưu:", os.listdir(SAVE_DIR))
print("Nạp lại: dựng lại encoder+HierAttnClassifier với cùng MODEL_NAME rồi model.load_state_dict(torch.load('model_state.pt')).")

Đã lưu: ['special_tokens_map.json', 'tokenizer.json', 'vocab.json', 'tokenizer_config.json', 'infer_config.json', 'model_state.pt', 'merges.txt']
Nạp lại: dựng lại encoder+HierAttnClassifier với cùng MODEL_NAME rồi model.load_state_dict(torch.load('model_state.pt')).


## 🔍 9. Inference thử

In [10]:
model.eval()

@torch.no_grad()
def predict_contract(code: str, threshold: float = None):
    threshold = best_t if threshold is None else threshold
    iid, am  = chunk_encode(code)
    n_chunks = torch.tensor([iid.shape[0]], dtype=torch.long)
    out = model(input_ids=iid.to(device), attention_mask=am.to(device), n_chunks=n_chunks.to(device))
    p   = torch.softmax(out.logits, dim=-1)[0, 1].item()
    return ("Vulnerable" if p >= threshold else "Safe"), p

TEST_CODE = """
function withdraw(uint amount) public {
    require(balances[msg.sender] >= amount);
    (bool success,) = msg.sender.call{value: amount}("");
    require(success);
    balances[msg.sender] -= amount;   // giảm số dư SAU khi gửi tiền -> reentrancy
}
"""
label, prob = predict_contract(TEST_CODE)
print(f"Dự đoán: {label}  (P(Vulnerable)={prob:.3f}, ngưỡng={best_t:.3f})")

Dự đoán: Safe  (P(Vulnerable)=0.516, ngưỡng=0.560)
